# 03 - Experiences de debiaisage

Comparaison baseline, reweighting et resampling sur la pile Python 3.14 du projet.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

for path in ['data/processed', 'models', 'outputs', 'reports', 'reports/figures']:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [2]:
def sanitize(obj):
    import numpy as np
    import math
    if isinstance(obj, dict):
        return {str(k): sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize(v) for v in obj]
    if isinstance(obj, tuple):
        return [sanitize(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return sanitize(obj.tolist())
    if isinstance(obj, np.generic):
        return sanitize(obj.item())
    if isinstance(obj, float) and not math.isfinite(obj):
        return None
    return obj

In [3]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression

from src.bias_mitigation import PreprocessingDebias
from src.data_processing import DataProcessor, generate_sample_data
from src.metrics import FairnessMetrics, PerformanceMetrics

DATA_PATH = PROJECT_ROOT / 'data/processed/dataset.csv'
if not DATA_PATH.exists():
    generate_sample_data(n_samples=5000, random_state=42).to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH)
protected_attrs = ['gender', 'race']
processor = DataProcessor(protected_attributes=protected_attrs, label_name='label')
df_encoded = processor.encode_categorical(df)
X_train, X_test, y_train, y_test = processor.split_data(df_encoded, test_size=0.2, random_state=42)
X_train_model = X_train.drop(columns=protected_attrs)
X_test_model = X_test.drop(columns=protected_attrs)

def evaluate(model):
    pred = model.predict(X_test_model)
    scores = model.predict_proba(X_test_model)[:, 1]
    result = {'performance': PerformanceMetrics.compute_metrics(y_test, pred, scores), 'fairness': {}}
    for attr in protected_attrs:
        values = X_test[attr].unique()
        fm = FairnessMetrics(attr, privileged_value=values[0], unprivileged_value=values[1] if len(values) > 1 else None)
        result['fairness'][attr] = fm.compute_all_metrics(y_test, pred, X_test[attr], scores)
    return result

baseline = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_model, y_train)
results = {'baseline': evaluate(baseline)}

primary_attr = 'gender'
values = X_train[primary_attr].unique()
debias = PreprocessingDebias(primary_attr, privileged_value=values[0], unprivileged_value=values[1] if len(values) > 1 else None)

weights = debias.reweighting(X_train_model, y_train, X_train[primary_attr])
reweighted = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_model, y_train, sample_weight=weights)
results['reweighting'] = evaluate(reweighted)

X_res, y_res, _ = debias.resampling_balance(X_train_model, y_train, X_train[primary_attr], strategy='oversample')
resampled = LogisticRegression(max_iter=1000, random_state=42).fit(X_res, y_res)
results['resampling'] = evaluate(resampled)

with open(PROJECT_ROOT / 'outputs/debiasing_results.json', 'w', encoding='utf-8') as f:
    json.dump(sanitize(results), f, indent=2, ensure_ascii=False)

rows = []
for method, payload in results.items():
    perf = payload['performance']
    fair = payload['fairness'][primary_attr]
    rows.append({
        'method': method,
        'accuracy': perf['accuracy'],
        'f1_score': perf['f1_score'],
        'disparate_impact': fair['disparate_impact'],
        'demographic_parity_difference': fair['demographic_parity_difference'],
    })
comparison = pd.DataFrame(rows)
display(comparison)

,method,accuracy,f1_score,disparate_impact,demographic_parity_difference
0,baseline,0.658,0.785445,1.040942,-0.037761
1,reweighting,0.646,0.782288,1.003520,-0.003432
2,resampling,0.657,0.779137,1.055762,-0.048774


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
comparison.set_index('method')[['accuracy', 'f1_score']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Performance')
axes[0].tick_params(axis='x', rotation=20)
comparison.set_index('method')[['disparate_impact', 'demographic_parity_difference']].plot(kind='bar', ax=axes[1])
axes[1].axhline(0.8, color='green', linestyle='--', linewidth=1)
axes[1].set_title('Fairness gender')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
fig.savefig(PROJECT_ROOT / 'reports/figures/debiasing_notebook_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)